# Download and Install PyFock and also PySCF (CPU and GPU)
### Clone latest development version of PyFock from GitHub repo

In [1]:
!git clone https://github.com/manassharma07/PyFock.git

Cloning into 'PyFock'...
remote: Enumerating objects: 6728, done.
remote: Counting objects: 100% (274/274), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 6728 (delta 144), reused 210 (delta 109), pack-reused 6454 (from 3)
Receiving objects: 100% (6728/6728), 372.53 MiB | 37.79 MiB/s, done.
Resolving deltas: 100% (4003/4003), done.
Updating files: 100% (3645/3645), done.


### Go to PyFock repo

In [2]:
import os
os.chdir('PyFock')

### Install PyFock

In [3]:
!pip3 install .

Processing /kaggle/working/PyFock
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.7 MB/s eta 0:00:00
  Created wheel for pyfock: filename=pyfock-0.1.4-py3-none-any.whl size=20369131 sha256=9a30a010698f6eb246cf7bb20c1a33b19fa0c1323c795fa5dce4d980d1e92585
  Stored in directory: /tmp/pip-ephem-wheel-cache-70a27dwf/wheels/b8/d0/ca/6d432daf38d95a1e34aee145f95b35c8e489dbb9d2b7fbe5fd
Successfully built pyfock


# Install pyscf CPU and GPU

In [4]:
!pip3 install pyscf 
!pip3 install gpu4pyscf-cuda12x

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 36.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.2/392.2 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.7/151.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 94.6 MB/s eta 0:00:00
  Created wheel for geometric: filename=geometric-1.1.1-py3-none-any.whl size=408349 sha256=7fdafabacc50c4c44ccf61759000fb653b42c3d585b4a32f6383c93fd08ab30e
  Stored in directory: /root/.cache/pip/wheels/e0/e4/26/f31cf16c0d7fa238c59215d5394d405c9749edf805e22232cf
Successfully built geometric


# Go inside the benchmark directory

In [5]:
os.chdir('benchmarks_tests/FINAL_Benchmarks_for_paper')

# Set environment variables

In [6]:
import os
import platform

# Set the number of threads/cores to be used by PyFock and PySCF
ncores = 2
os.environ['OMP_NUM_THREADS'] = str(ncores)
os.environ["OPENBLAS_NUM_THREADS"] = str(ncores) # export OPENBLAS_NUM_THREADS=4 
os.environ["MKL_NUM_THREADS"] = str(ncores) # export MKL_NUM_THREADS=4
os.environ["VECLIB_MAXIMUM_THREADS"] = str(ncores) # export VECLIB_MAXIMUM_THREADS=4
os.environ["NUMEXPR_NUM_THREADS"] = str(ncores) # export NUMEXPR_NUM_THREADS=1



# Check if the environment variables are properly set
print("Number of cores being actually used/requested for the benchmark:", ncores)
print('Confirming that the environment variables are properly set...')
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', None))
print('OPENBLAS_NUM_THREADS =', os.environ.get('OPENBLAS_NUM_THREADS', None))
print('MKL_NUM_THREADS =', os.environ.get('MKL_NUM_THREADS', None))
print('VECLIB_MAXIMUM_THREADS =', os.environ.get('VECLIB_MAXIMUM_THREADS', None))
print('NUMEXPR_NUM_THREADS =', os.environ.get('NUMEXPR_NUM_THREADS', None))





Number of cores being actually used/requested for the benchmark: 2
Confirming that the environment variables are properly set...
OMP_NUM_THREADS = 2
OPENBLAS_NUM_THREADS = 2
MKL_NUM_THREADS = 2
VECLIB_MAXIMUM_THREADS = 2
NUMEXPR_NUM_THREADS = 2


In [7]:
!export CUDA_VISIBLE_DEVICES=0

# PySCF Benchmarks (these will include grid gen timings)

In [8]:
# Run your tasks here
from timeit import default_timer as timer
import numpy as np

from pyscf import gto
from gpu4pyscf.dft import rks

# RI-DFT SCF benchmark


# GGA_X_PBE, GGA_C_PBE (PBE)
funcx = 101
funcc = 130


funcidpyscf = str(funcx)+','+str(funcc)


basis_set_name = 'def2-SVP'


auxbasis_name = 'def2-universal-jfit'

# List of all molecules to calculate
xyz_files = [
    'Structures/water_cluster_5.xyz',
    'Structures/water_cluster_10.xyz',
    'Structures/water_cluster_20.xyz',
    'Structures/water_cluster_32.xyz',
    'Structures/water_cluster_47.xyz',
    # 'Structures/water_cluster_76.xyz',
    # 'Structures/water_cluster_100.xyz',
    # 'Structures/water_cluster_139.xyz',
]


# Loop through all molecules
for xyzFilename in xyz_files:
    print(f"\n{'='*60}")
    print(f"Processing: {xyzFilename}")
    print(f"{'='*60}")

    # ---------PySCF---------------
    #Comparison with PySCF
    molPySCF = gto.Mole()
    molPySCF.atom = xyzFilename
    molPySCF.basis = basis_set_name
    molPySCF.cart = False
    # molPySCF.verbose = 6
    #molPySCF.max_memory=250000
    molPySCF.build()

    print('\n\nPySCF Results\n\n')
    mf = rks.RKS(molPySCF).density_fit(auxbasis=auxbasis_name)
    mf.xc = funcidpyscf
    mf.verbose = 6
    #mf.direct_scf = True
    mf.init_guess = 'minao'
    dmat_init = mf.init_guess_by_minao(molPySCF)
    mf.max_cycle = 50
    mf.conv_tol = 1e-7
    mf.grids.level = 3
    start=timer()
    energyPyscf = mf.kernel(dm0=dmat_init)
    print('Nuc-Nuc PySCF= ', molPySCF.energy_nuc())
    print('One electron integrals energy',mf.scf_summary['e1'])
    print('Coulomb energy ',mf.scf_summary['coul'])
    print('EXC ',mf.scf_summary['exc'])
    duration = timer()-start
    print('PySCF time: ', duration)
    print('PySCF Grid Size: ', mf.grids.weights.shape)

/usr/local/lib/python3.12/dist-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')



Processing: Structures/water_cluster_5.xyz


PySCF Results




******** <class 'gpu4pyscf.df.df_jk.DFRKS'> ********
method = DFRKS
initial guess = minao
damping factor = None
level_shift factor = None
DIIS = <class 'gpu4pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-07
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = False
XC library gpu4pyscf.dft.libxc version 7.0.0 (CUDA)
    unable to decode the reference due to https://github.com/NVIDIA/cuda-python/issues/29
XC functionals = 101,130
small_rho_cutoff = 0
Set gradient conv threshold to 0.000316228
    CPU time for hcore                                                   4.20 sec, wall time      4.33 sec, GPU time   4329.17 ms
    CPU time for setting up grids                                        1.24 sec, wall time      1.25 sec, GPU time   1245.97 ms
nelec by numeric integration = 49.90596261778306
    CPU time for vxc                                                     7.19 sec

# PyFock Benchmarks
These will use grids generated by PySCF

In [9]:
from pyfock import Basis
from pyfock import Mol
from pyfock import Integrals
from pyfock import DFT
from timeit import default_timer as timer
import numpy as np

from pyscf import gto, dft, df, scf

#RI-DFT SCF benchmark 


# GGA_X_PBE, GGA_C_PBE (PBE)
funcx = 101
funcc = 130


funcidcrysx = [funcx, funcc]
funcidpyscf = str(funcx)+','+str(funcc)


basis_set_name = 'def2-SVP'


auxbasis_name = 'def2-universal-jfit'

# List of all molecules to calculate
xyz_files = [
    'Structures/water_cluster_5.xyz',
    'Structures/water_cluster_10.xyz',
    'Structures/water_cluster_20.xyz',
    'Structures/water_cluster_32.xyz',
    'Structures/water_cluster_47.xyz',
    # 'Structures/water_cluster_76.xyz',
    # 'Structures/water_cluster_100.xyz',
    # 'Structures/water_cluster_139.xyz',
]


# Loop through all molecules
for xyzFilename in xyz_files:
    print(f"\n{'='*60}")
    print(f"Processing: {xyzFilename}")
    print(f"{'='*60}")

    # ---------PySCF---------------
    #Comparison with PySCF
    molPySCF = gto.Mole()
    molPySCF.atom = xyzFilename
    molPySCF.basis = basis_set_name
    molPySCF.cart = False
    # molPySCF.verbose = 4
    molPySCF.build()

    print('\n\nPySCF Results\n\n')
    start=timer()
    mf = dft.rks.RKS(molPySCF).density_fit(auxbasis=auxbasis_name)
    mf.xc = funcidpyscf
    mf.verbose = 4
    mf.direct_scf = False
    mf.init_guess = 'minao'
    dmat_init = mf.init_guess_by_minao(molPySCF)
    mf.max_cycle = 1
    mf.conv_tol = 1e-7
    mf.grids.level = 3
    energyPyscf = mf.kernel(dm0=dmat_init)
    duration = timer()-start
    print('PySCF time: ', duration)
    pyscfGrids = mf.grids
    print('PySCF Grid Size: ', pyscfGrids.weights.shape)
    mf = 0

    #--------------------CrysX --------------------------

    #Initialize a Mol object with somewhat large geometry
    molCrysX = Mol(coordfile=xyzFilename)

    #Initialize a Basis object with a very large basis set
    basis = Basis(molCrysX, {'all':Basis.load(mol=molCrysX, basis_name=basis_set_name)})

    auxbasis = Basis(molCrysX, {'all':Basis.load(mol=molCrysX, basis_name=auxbasis_name)})

    dftObj = DFT(molCrysX, basis, auxbasis, xc=funcidcrysx, grids=pyscfGrids)
    dftObj.dmat = dmat_init
    dftObj.conv_crit = 1e-7
    dftObj.max_itr = 50
    dftObj.ncores = ncores
    dftObj.save_ao_values = False
    dftObj.rys = True
    dftObj.DF_algo = 10
    dftObj.blocksize = 20480
    dftObj.XC_algo = 3
    dftObj.debug = False
    dftObj.sortGrids = False
    dftObj.xc_bf_screen = True
    dftObj.threshold_schwarz = 1e-10
    dftObj.strict_schwarz = False
    dftObj.cholesky = False
    dftObj.orthogonalize = True
    # SAO or CAO basis
    dftObj.sao = True

    # GPU acceleration
    dftObj.use_gpu = True
    dftObj.keep_ao_in_gpu = False
    dftObj.use_libxc = False
    dftObj.n_streams = 1 # Changing this to anything other than 1 won't make any difference 
    dftObj.n_gpus = 1 # Specify the number of GPUs
    dftObj.free_gpu_mem = True
    dftObj.threads_x = 32
    dftObj.threads_y = 32
    dftObj.dynamic_precision = True
    dftObj.keep_ints3c2e_in_gpu = True


    # Using PySCF grids to compare the energies
    energyCrysX, dmat = dftObj.scf()


Processing: Structures/water_cluster_5.xyz


PySCF Results


Initial guess from minao.


******** <class 'pyscf.df.df_jk.DFRKS'> ********
method = DFRKS
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-07
SCF conv_tol_grad = None
SCF max_cycles = 1
direct_scf = False
chkfile to save SCF result = /tmp/tmpj2gjg8uj
max_memory 4000 MB (current use 1260 MB)
XC library pyscf.dft.libxc version 7.0.0
    S. Lehtola, C. Steigemann, M. J.T. Oliveira, and M. A.L. Marques.,  SoftwareX 7, 1–5 (2018)
XC functionals = 101,130
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 77, 3865 (1996)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 78, 1396 (1997)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 77, 3865 (1996)
    J. P. Perdew, K. Burke, and M. Ernzerhof.,  Phys. Rev. Lett. 78, 1396 (1997)
small_rho_cutoff = 0
Set gradient con

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 48 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 48 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: Num

Core H size in GB  0.0001
done!
Time taken 22.63 seconds.



/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))




User requested to use Rys Quadrature Algorithm for the evaluation of ERIs
Stricter version of Schwarz screening:  False

Calculating three centered two electron and two-centered two-electron integrals...


Time taken for two-centered two-electron integrals 15.14 seconds.



Performing Schwarz screening...
Threshold  1e-10


...

Number of electrons:  99.90868



------Iteration 1--------


Energies (in Hartrees)

Electron-Nuclear Energy        -3085.6424009087486
Nuclear repulsion Energy         647.7065461172319
Kinetic Energy                   758.1524577075539
Coulomb Energy                  1008.4625446194498
Exchange-Correlation Energy      -90.8507614135742
--------------------------------------------------
Total Energy                    -762.1716138780873
--------------------------------------------------



Energy difference :  762.1716138780873


HOMO-LUMO gap: 0.3198857420216099 au
HOMO-LUMO gap: 8.705 eV


Time taken for the previous iteration: 0.38 seconds 



Density matrix difference norm:  4.339554290334106 





XC algo 3
Number of electrons:  100.00001



------Iteration 2--------


Energies (in Hartrees)

Electron-Nuclear Energy        -3141.3882315970168
Nuclear repulsion Energy         647.7065461172319
Kinetic Energy                   776.9490539547178
Coulomb Energy                  1050

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 36 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Number of electrons:  199.85269



------Iteration 1--------


Energies (in Hartrees)

Electron-Nuclear Energy        -8153.6283467325657
Nuclear repulsion Energy        2286.8096438306616
Kinetic Energy                  1516.5611547812803
Coulomb Energy                  3007.4412360456363
Exchange-Correlation Energy     -181.9052581787109
--------------------------------------------------
Total Energy                   -1524.7215702536987
--------------------------------------------------



Energy difference :  1524.7215702536987


HOMO-LUMO gap: 0.30789592731855764 au
HOMO-LUMO gap: 8.378 eV


Time taken for the previous iteration: 0.71 seconds 



Density matrix difference norm:  6.139727763205927 





XC algo 3
Number of electrons:  200.00003



------Iteration 2--------


Energies (in Hartrees)

Electron-Nuclear Energy        -8263.0828552830026
Nuclear repulsion Energy        2286.8096438306616
Kinetic Energy                  1552.9360136337134
Coulomb Energy                  3

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 98 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))






XC algo 3
Number of electrons:  319.77927



------Iteration 1--------


Energies (in Hartrees)

Electron-Nuclear Energy       -15923.5782454208893
Nuclear repulsion Energy        5098.4401211606146
Kinetic Energy                  2426.6112650220120
Coulomb Energy                  6249.9715873585174
Exchange-Correlation Energy     -291.1129150390625
--------------------------------------------------
Total Energy                   -2439.6681869188069
--------------------------------------------------



Energy difference :  2439.668186918807


HOMO-LUMO gap: 0.30096313640041206 au
HOMO-LUMO gap: 8.190 eV


Time taken for the previous iteration: 1.46 seconds 



Density matrix difference norm:  7.758761229431994 





XC algo 3
Number of electrons:  320.00006



------Iteration 2--------


Energies (in Hartrees)

Electron-Nuclear Energy       -16100.5020070235551
Nuclear repulsion Energy        5098.4401211606146
Kinetic Energy                  2484.2017033302050
Coulomb Energy      